In [44]:
import torch
from transformers import  CamembertForMaskedLM, CamembertTokenizer, CamembertModel,RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline
from transformers import BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM 
from transformers import TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

In [45]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)

In [46]:
#Data
source = "test/Polish.txt"
file = open(source, "r", encoding = 'utf-8')
lines = file.readlines()

data = []

for l in range(12):
    line = lines[l]
    
    data.append(line)

In [47]:
def bert_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = "[CLS] " + text + " [SEP]" #special token for BERT, RoBERTa
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    length = len(tokenized_text)-2
    for masked_index in range(1,len(tokenized_text)-1):
        # Mask a token that we will try to predict back with `BertForMaskedLM`
        masked_word = tokenized_text[masked_index]
        #tokenized_text[masked_index] = '<mask>' #special token for XLNet
        tokenized_text[masked_index] = '[MASK]' #special token for BERT, RoBerta
        # Convert token to vocabulary indices
        indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
        index = torch.tensor(tokenizer.convert_tokens_to_ids(masked_word))
        tokens_tensor = torch.tensor([indexed_tokens])
        tokens_tensor = tokens_tensor.to('cuda')
        index = index.to('cuda')
        #masked_tensor = torch.tensor([masked_index])
        with torch.no_grad():
            outputs = model(tokens_tensor.to('cuda'))
        prediction_scores = outputs[0]
        prediction_scores = prediction_scores.view(-1, model.config.vocab_size)
        prediction_scores = prediction_scores[masked_index].unsqueeze(0)
        loss_fct = CrossEntropyLoss(ignore_index=-1)  # -1 index = padding token
        masked_lm_loss = loss_fct(prediction_scores, index.view(-1))
        tokenized_text[masked_index] = masked_word
        sentence_score -= masked_lm_loss.item()
        tokenized_text[masked_index] = masked_word
    sentence_score = sentence_score/length
    return sentence_score

In [48]:
def uni_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = text
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    length = len(tokenized_text)
    tokens_tensor = torch.tensor([indexed_tokens])
    tokens_tensor = tokens_tensor.to('cuda')
    #masked_tensor = torch.tensor([masked_index])
    with torch.no_grad():
        outputs = model(tokens_tensor, labels= tokens_tensor)
    loss = outputs[0]
    sentence_score = -loss
    return sentence_score

In [61]:
def score_model(model, tokenizer, data):
    opts = ["stały","dzieci","dalszy","trudno","bawi","krajach","pieniądze","kosztowne","młode","wierzą","osiąga","trenowania", "stały","dzieci"]
    #opts = ["vyrástli","deti","druhoradé","ťažké","hrá","krajinách","peniaze","drahé","nevyvinuté","veria","dostane","trénovania"]
    df = pd.DataFrame()
    for d in tqdm(data):
        print(d)
        print("Correct option is: ", opts[data.index(d)])
        scores = {}
        for o in opts:
            sentence = d.replace("{}", o)
            scores.update({o : float(uni_predict(sentence, model, tokenizer).item())})
        df = df.append(scores, ignore_index=True)
        scores = sorted(scores.items(), key=lambda x: x[1], reverse = True)
        for key, value in scores:
            print(key, ':', value)
        print()
        
    print(df)
    df = df.apply(lambda row: row / row.mean(), axis=1)
    x,y = linear_sum_assignment(df)
    out = pd.DataFrame({'Word': df.columns[y], 'Sentence': df.index[x]})
    print(out)

In [62]:
#Polish
#print(torch.cuda.is_available())
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
#print(bert_predict("text text text", model, tokenizer))
scoredModel = score_model(model, tokenizer, data)

Some weights of the model checkpoint at dkleczek/bert-base-polish-uncased-v1 were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


  0%|          | 0/12 [00:00<?, ?it/s]

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały


C:\Users\Secon\AppData\Local\Temp\ipykernel_2640\3359511908.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_2640\3359511908.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)


stały : -0.1063566654920578
osiąga : -0.11348798871040344
trenowania : -0.27235785126686096
bawi : -0.28937220573425293
dzieci : -0.39617660641670227
wierzą : -0.48057010769844055
pieniądze : -0.5180937051773071
młode : -0.5759477615356445
krajach : -0.6266823410987854
kosztowne : -0.6311476230621338
trudno : -0.646591305732727
dalszy : -0.8302435874938965

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
młode : -0.22091928124427795
dzieci : -0.23963668942451477
trenowania : -0.2647949457168579
pieniądze : -0.2696557641029358
krajach : -0.2925722599029541
osiąga : -0.3072097599506378
wierzą : -0.34755390882492065
trudno : -0.6901698708534241
dalszy : -0.911700963973999
kosztowne : -1.0733739137649536
bawi : -1.181312084197998
stały : -1.2786576747894287

Praca w szkole, spotykanie się ze znajomymi I inne zainteresowania muszą zejść na {} plan.

Correct option is:  dalszy


...

C:\Users\Secon\AppData\Local\Temp\ipykernel_2640\3359511908.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_2640\3359511908.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)


wierzą : -0.3044643998146057
trenowania : -0.3097795248031616
osiąga : -0.314588725566864
dzieci : -0.36695200204849243
pieniądze : -0.3705175518989563
trudno : -0.4543890357017517
młode : -0.4609788656234741
stały : -0.5484269261360168
kosztowne : -0.5902099013328552
dalszy : -0.6480005979537964
krajach : -1.0707582235336304
bawi : -1.0911314487457275

Odzież sportowa, dowozy na zawody, specjalistyczne wyposażenie itp. mogą być bardzo {}.

Correct option is:  kosztowne
wierzą : -0.3625321090221405
kosztowne : -0.4119102656841278
młode : -0.4196607172489166
trenowania : -0.45334330201148987
dzieci : -0.5046951174736023
trudno : -0.5130375027656555
osiąga : -0.5633575320243835
stały : -0.8392831683158875
bawi : -1.0428481101989746
pieniądze : -1.054648995399475
krajach : -1.4811159372329712
dalszy : -1.6197965145111084

W zrozumiały sposób wielu rodziców niepokoi się, że to niebezpieczne, żeby zaczynać poważne treningi sportowe w tak wczesnym wieku. Niektórzy lekarze przyznają, że {} mi

C:\Users\Secon\AppData\Local\Temp\ipykernel_2640\3359511908.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_2640\3359511908.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)


trenowania : -0.08340142667293549
osiąga : -0.08555501699447632
dzieci : -0.08969287574291229
młode : -0.09208637475967407
wierzą : -0.10673309862613678
kosztowne : -0.11173252016305923
pieniądze : -0.20994555950164795
stały : -0.21721714735031128
trudno : -0.23029974102973938
dalszy : -0.4200015962123871
krajach : -0.42695608735084534
bawi : -0.5098330974578857

Jednakże trenerzy {}, że można osiągnąć szczyt jako sportowiec jedynie jeśli zaczyna się młodo.

Correct option is:  wierzą
wierzą : -0.21194608509540558
osiąga : -0.3357166349887848
trenowania : -0.3645811975002289
dzieci : -0.47966018319129944
trudno : -0.5547431707382202
pieniądze : -0.7712215781211853
stały : -0.8336604237556458
młode : -0.8962202072143555
bawi : -0.9449784159660339
krajach : -0.9583176374435425
kosztowne : -1.2085590362548828
dalszy : -1.2249596118927002

Jasne jest, że bardzo niewiele osób {} szczyty. 

Correct option is:  osiąga
osiąga : -0.6676719188690186
stały : -0.6884061694145203
dalszy : -0.715557

C:\Users\Secon\AppData\Local\Temp\ipykernel_2640\3359511908.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_2640\3359511908.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)


In [17]:
#Czech 2
#Uses BERT tokenizer to avoid sentencepiece "not a string" error
tokenizer = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
model = AlbertForMaskedLM.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True).cuda()
scoredModel = score_model(model, tokenizer, data)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
All TF 2.0 model weights were used when initializing AlbertForMaskedLM.

Some weights of AlbertForMaskedLM were not initialized from the TF 2.0 model and are newly initialized: ['predictions.decoder.weight', 'predictions.decoder.bias', 'predictions.decoder.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
trudno : tensor(-0.6725, device='cuda:0')
bawi : tensor(-0.6728, device='cuda:0')
kosztowne : tensor(-0.6744, device='cuda:0')
krajach : tensor(-0.6802, device='cuda:0')
trenowania : tensor(-0.6828, device='cuda:0')
pieniądze : tensor(-0.6922, device='cuda:0')
dalszy : tensor(-0.6976, device='cuda:0')
młode : tensor(-0.6997, device='cuda:0')
osiąga : tensor(-0.7263, device='cuda:0')
stały : tensor(-0.7270, device='cuda:0')
dzieci : tensor(-0.7315, device='cuda:0')
wierzą : tensor(-0.7505, device='cuda:0')

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
pieniądze : tensor(-1.0340, device='cuda:0')
stały : tensor(-1.0980, device='cuda:0')
wierzą : tensor(-1.0980, device='cuda:0')
dalszy : tensor(-1.1636, device='c

RuntimeError: The expanded size of the tensor (520) must match the existing size (512) at non-singleton dimension 1.  Target sizes: [1, 520].  Tensor sizes: [1, 512]

In [18]:
tokenizer = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
model = RobertaForMaskedLM.from_pretrained('gerulata/slovakbert').cuda()
scoredModel = score_model(model, tokenizer, data)

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
bawi : tensor(-0.5924, device='cuda:0')
stały : tensor(-0.6042, device='cuda:0')
kosztowne : tensor(-0.6042, device='cuda:0')
pieniądze : tensor(-0.6159, device='cuda:0')
dalszy : tensor(-0.6219, device='cuda:0')
trudno : tensor(-0.6245, device='cuda:0')
dzieci : tensor(-0.6308, device='cuda:0')
trenowania : tensor(-0.6530, device='cuda:0')
wierzą : tensor(-0.6578, device='cuda:0')
młode : tensor(-0.6780, device='cuda:0')
osiąga : tensor(-0.7316, device='cuda:0')
krajach : tensor(-0.7799, device='cuda:0')

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
pieniądze : tensor(-1.0840, device='cuda:0')
dalszy : tensor(-1.1505, device='cuda:0')
osiąga : tensor(-1.1516, device='cuda:0')
wierzą : tensor(-1.1555, device='

RuntimeError: The expanded size of the tensor (564) must match the existing size (514) at non-singleton dimension 1.  Target sizes: [1, 564].  Tensor sizes: [1, 514]

In [19]:
tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-1.4B")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-1.4B").cuda()
scoredModel = score_model(model, tokenizer, data)

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
stały : tensor(-2.7270, device='cuda:0')
wierzą : tensor(-2.8120, device='cuda:0')
pieniądze : tensor(-2.8189, device='cuda:0')
dzieci : tensor(-2.8311, device='cuda:0')
trudno : tensor(-2.8629, device='cuda:0')
kosztowne : tensor(-2.8890, device='cuda:0')
młode : tensor(-2.8895, device='cuda:0')
trenowania : tensor(-2.8931, device='cuda:0')
dalszy : tensor(-2.9277, device='cuda:0')
osiąga : tensor(-2.9297, device='cuda:0')
bawi : tensor(-2.9317, device='cuda:0')
krajach : tensor(-3.0210, device='cuda:0')

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
młode : tensor(-3.2715, device='cuda:0')
dzieci : tensor(-3.3320, device='cuda:0')
pieniądze : tensor(-3.3950, device='cuda:0')
osiąga : tensor(-3.4117, device='c

stały : tensor(-2.6697, device='cuda:0')
pieniądze : tensor(-2.6835, device='cuda:0')
wierzą : tensor(-2.6862, device='cuda:0')
dzieci : tensor(-2.6887, device='cuda:0')
młode : tensor(-2.6906, device='cuda:0')
trudno : tensor(-2.6933, device='cuda:0')
trenowania : tensor(-2.6989, device='cuda:0')
kosztowne : tensor(-2.6991, device='cuda:0')
dalszy : tensor(-2.7005, device='cuda:0')
bawi : tensor(-2.7027, device='cuda:0')
osiąga : tensor(-2.7051, device='cuda:0')
krajach : tensor(-2.7171, device='cuda:0')

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by _ się one najlepszymi sportowcami?  Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku. Praca w szkole, spotykanie się ze znajomymi I inne zainteresowania muszą zejść na _ plan. Bardzo _ jest wyjaśnić małym dzieciom dlaczego muszą trenować pięć godzin dziennie. To dotyczy także weekendów, kiedy większość 

In [9]:
def permute_score():#model, tokenizer, data):
    def permute(array, current_index=0):
        if current_index == len(array) - 1:
            #print(array)
            x=0
        else:
            for i in range(current_index, len(array)):
                array[current_index], array[i] = array[i], array[current_index]
                permute(array, current_index + 1)
                array[current_index], array[i] = array[i], array[current_index]
                
    opts = ["stały","dzieci","dalszy","trudno","bawi","krajach","pieniądze","kosztowne","młode","wierzą","osiąga","trenowania", "stały","dzieci"]
    permute(opts)
    
    

                
permute_score()

KeyboardInterrupt: 